In [1]:
import geopandas as gpd

lith = gpd.read_file(
r"D:\landslide\tehri_landslide\data\processed\lithology\lithology_tehri.shp"
)

def classify_lithology(text):

    text = text.upper()

    if "GRANITE" in text or "GNEISS" in text:
        return 1

    elif "SCHIST" in text or "PHYLLITE" in text:
        return 2

    elif "QUARTZITE" in text:
        return 3

    elif "LIMESTONE" in text or "DOLOMITE" in text:
        return 4

    elif "SHALE" in text or "SLATE" in text:
        return 5

    elif "AMPHIBOLITE" in text or "DOLERITE" in text or "EPIDIORITE" in text:
        return 6

    elif "SAND" in text or "CLAY" in text or "GRAVEL" in text:
        return 7

    else:
        return 8
        

lith["lith_class"] = lith["lithologic"].apply(classify_lithology)

print(lith["lith_class"].value_counts())

lith_class
3    122
6     86
4     83
2     71
7     49
1     30
8     28
5     18
Name: count, dtype: int64


In [2]:
lith.to_file(
r"D:\landslide\tehri_landslide\data\processed\lithology\lithology_grouped.shp"
)

In [6]:
tehri = gpd.read_file(
r"F:\New folder (2)\tehri_new\tehri.shp"
)

lith_clip = gpd.overlay(lith, tehri, how="intersection")

print("Features after clip:", len(lith_clip))

Features after clip: 487


In [7]:
lith_clip.to_file(
r"D:\landslide\tehri_landslide\data\processed\lithology\lithology_tehri_clip.shp"
)

In [8]:
import rasterio
from rasterio.features import rasterize

ref = r"D:\landslide\final_data\Slope_deg_10m.tif"

with rasterio.open(ref) as src:
    transform = src.transform
    shape = (src.height, src.width)
    crs = src.crs

lith_clip = gpd.read_file(
r"D:\landslide\tehri_landslide\data\processed\lithology\lithology_tehri_clip.shp"
)

shapes = [(geom, value) for geom, value in zip(lith_clip.geometry, lith_clip.lith_class)]

lith_raster = rasterize(
    shapes,
    out_shape=shape,
    transform=transform,
    fill=0,
    dtype="int16"
)

In [9]:
out_path = r"D:\landslide\final_data\lithology_tehri_10m.tif"

with rasterio.open(
    out_path,
    "w",
    driver="GTiff",
    height=shape[0],
    width=shape[1],
    count=1,
    dtype="int16",
    crs=crs,
    transform=transform
) as dst:
    dst.write(lith_raster, 1)

In [10]:
with rasterio.open(out_path) as src:
    arr = src.read(1)

import numpy as np
print("Unique lithology classes:", np.unique(arr))

Unique lithology classes: [0 1 2 3 4 5 6 7 8]
